# Description

W tym notatniku przeprowadzane są wszelkie eksperymenty, zarówno dla autoenkodera wariacyjnego i nie wariacyjnego, dla wszystkich członów funkcji straty, w wersji z douczaniem i bez (łącznie 12 eksperymentów)

# Imports

In [51]:
IS_NEW_APPROACH = True

In [52]:
%load_ext autoreload
%autoreload 2
import torch
from torch.utils.data import Dataset, DataLoader, random_split
import pytorch_lightning as pl
from pytorch_lightning.loggers import WandbLogger
from pytorch_lightning.callbacks import RichProgressBar
import yaml
import sys
import os
import tqdm
import wandb
import json
import random

sys.path.append('../')  # Dodajemy katalog wyżej, żeby src był widoczny
from src.models.KlejdaGAE.KlejdaGraphAutoencoder import KlejdaGraphAutoencoder
from src.models.KlejdaGAE.KlejdaVariationalGraphAutoencoder import KlejdaVariationalGraphAutoencoder
from src.models.NewGAE.GraphAutoencoder import GraphAutoencoder
from src.models.NewGAE.VariationalGraphAutoencoder import VariationalGraphAutoencoder
from utils.FramsticksGraphDataset import FramsticksGraphDataset
from pyprojroot import here

current_dir = os.getcwd()
framspy_path = os.path.abspath(os.path.join(current_dir, '..', 'external', 'framspy'))
if framspy_path not in sys.path:
	sys.path.insert(0, framspy_path)
from FramsticksLib import FramsticksLib
from deap import tools, algorithms
import yaml
from src.deap.deap_setup import prepare_native_toolbox, prepare_cmaes_toolbox
from src.deap.constraints import is_feasible_fitness_criteria
from src.deap.save_and_load_results import save_genotypes_json
from src.deap.custom_ea_algorithms import run_cma_es_with_validation
from utils.FramsticksGraphDataset import FramsticksGraphDataset
from src.deap.AutoencoderEvaluator import AutoencoderEvaluator
import numpy as np
import frams
from copy import deepcopy

import time

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Pre-processing

In [53]:
project_dir = here()
# Przygotowanie checkpointów nauczonych autoenkoderów
checkpoints_dir = project_dir / 'notebooks' / 'checkpoints' / 'final_checkpoints'
if IS_NEW_APPROACH:
	checkpoints_dir = checkpoints_dir / 'new'
else:
	checkpoints_dir = checkpoints_dir / 'klejda'

checkpoint_gae = torch.load(checkpoints_dir / 'gae.ckpt')
checkpoint_vgae = torch.load(checkpoints_dir / 'vgae.ckpt')

CONTINUAL_TRAINING_DATASET_VERSION = 'remove_worse'

wandb.login()

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


False

In [54]:
# Przygotowanie konfiguracji dla gae
configs_dir = project_dir / 'configs'
if IS_NEW_APPROACH:
	config_gae_path = configs_dir / 'gae_config_large.yaml'
else:
	config_gae_path = configs_dir / 'klejda_gae_config.yaml'
with open(config_gae_path) as f:
	config_gae = yaml.safe_load(f)
	config_vgae = config_gae

In [55]:
# Wersja CMA-ES
with open("../configs/final_evolution_config.yaml", 'r') as f:
	evolution_config = yaml.safe_load(f)
frams.init(
	evolution_config['frams_path']
)
frams_lib = FramsticksLib(evolution_config['frams_path'], evolution_config['frams_lib'], evolution_config['sim_file'])
toolbox = prepare_cmaes_toolbox(frams_lib, evolution_config)

hof = tools.HallOfFame(evolution_config['hof_size'])
stats = tools.Statistics(lambda ind: ind.fitness.values)


def safe_filter_feasible(func, criteria):
	feasible_fits = list(filter(is_feasible_fitness_criteria, criteria))
	if len(feasible_fits) == 0:
		return np.nan
	return round(func(feasible_fits), 3)


stats.register("min", lambda fit: safe_filter_feasible(np.min, fit))
stats.register("avg", lambda fit: safe_filter_feasible(np.mean, fit))
stats.register("max", lambda fit: safe_filter_feasible(np.max, fit))

Using Framsticks version: 5.5
Home (writable) dir     : C:\Users\witek\PycharmProjects\Magisterka\external\Framsticks55\data
Resources dir           : C:\Users\witek\PycharmProjects\Magisterka\external\Framsticks55\data

Using Framsticks version: 5.5
Home (writable) dir     : C:\Users\witek\PycharmProjects\Magisterka\external\Framsticks55\data
Resources dir           : C:\Users\witek\PycharmProjects\Magisterka\external\Framsticks55\data

Available objects: ['CheckpointEvent', 'Collision', 'CrCollision', 'Creature', 'CreatureSettings', 'CreatureSignals', 'CreatureSnapshot', 'Dictionary', 'ExpProperties', 'ExpState', 'ExtValue', 'File', 'FunctionReference', 'GenMan', 'GenManStats', 'GenePool', 'GenePools', 'Geno', 'GenoConverters', 'Genotype', 'Interface', 'Joint', 'Loader', 'Math', 'MechJoint', 'MechPart', 'MessageCatcher', 'Model', 'ModelGeometry', 'ModelSymmetry', 'Neuro', 'NeuroClass', 'NeuroClassLibrary', 'NeuroDef', 'NeuroSignals', 'NeuronsSimEnabled', 'ODE', 'Orient', 'Part', 'Pop

# Experiments

In [56]:
def update_genotypes_buffer(current_genotypes, new_genotypes, mode, max_size):
	if mode == 'append':
		return current_genotypes + new_genotypes

	combined = current_genotypes + new_genotypes

	if len(combined) <= max_size:
		return combined

	num_to_remove = len(combined) - max_size

	if mode == 'remove_worse':
		combined.sort(key=lambda x: x['fitness'], reverse=True)
		return combined[:max_size]

	elif mode == 'remove_last':
		return combined[num_to_remove:]

	elif mode == 'remove_random':
		return random.sample(combined, max_size)
	else:
		raise ValueError(
			f"Nieznana strategia: {mode}. Dostępne: 'append', 'remove_worse', 'remove_last', 'remove_random'")

## GAE

In [7]:
if IS_NEW_APPROACH:
	gae_non_cyclic = GraphAutoencoder(config=config_gae, frams_module=frams).double()
else:
	gae_non_cyclic = KlejdaGraphAutoencoder(config=config_gae, frams_module=frams).double()
gae_non_cyclic.load_state_dict(checkpoint_gae['state_dict'])
gae_non_cyclic.eval()
evaluator = AutoencoderEvaluator(gae_non_cyclic, frams_lib, evolution_config['opt_criteria'], evolution_config)
toolbox.register("evaluate", evaluator)

### Trained once

In [8]:
pop, log, reconstruction_ratio, _ = run_cma_es_with_validation(
	toolbox,
	ngen=evolution_config['generations'],
	stats=stats,
	halloffame=hof,
	verbose=True
)

print(f"\nNajlepszy fitness w HoF: {hof[0].fitness.values[0]}")
save_genotypes_json(evolution_config["result_filepath"], hof)

RuntimeError: mat1 and mat2 shapes cannot be multiplied (1x3 and 15x64)

### Continual training

In [39]:
with open("../configs/klejda_gae_config.yaml") as f:
    config_gae = yaml.safe_load(f)

loaded_genotypes = []
with open("../results/sampled_best_individuals_new_mini_merged.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line.strip())
        loaded_genotypes.append(obj)

genotypes = deepcopy(loaded_genotypes)
BUFFER_MAX_SIZE = len(genotypes)

for i in range(3):
    toolbox = prepare_cmaes_toolbox(frams_lib, evolution_config)
    toolbox.register("evaluate", evaluator)

    pop, log, created_individuals = run_cma_es_with_validation(
        toolbox,
        ngen=evolution_config['generations'],
        stats=stats,
        halloffame=hof,
        verbose=True,
    )

    new_inds = [{'genotype': ind.genotype, 'fitness': list(ind.fitness.values)} for ind in created_individuals]

    print(f"Dodatkowi osobnicy: {len(new_inds)}")
    fitnesses = [ind['fitness'][0] if isinstance(ind['fitness'], (list, tuple)) else ind['fitness'] for ind in genotypes]
    print(f"Statystyki przed: min: {min(fitnesses)}, max: {max(fitnesses)}, mean: {sum(fitnesses)/len(fitnesses)}")

    genotypes = update_genotypes_buffer(
        current_genotypes=genotypes,
        new_genotypes=new_inds,
        mode=CONTINUAL_TRAINING_DATASET_VERSION,
        max_size=BUFFER_MAX_SIZE
    )
    fitnesses = [ind['fitness'][0] if isinstance(ind['fitness'], (list, tuple)) else ind['fitness'] for ind in genotypes]
    print(f"Statystyki po: min: {min(fitnesses)}, max: {max(fitnesses)}, mean: {sum(fitnesses)/len(fitnesses)}")

    dataset = FramsticksGraphDataset(genotypes, config_gae["max_nodes"])

    dataset_size = len(dataset)
    train_size = int(0.8 * dataset_size)
    val_size = dataset_size - train_size
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

    wandb_logger = WandbLogger(
        project="Framsticks-GAE",
        name=f"GAE-Continual-{CONTINUAL_TRAINING_DATASET_VERSION}-Iter-{i}",
        save_dir=config_gae["save_dir"]
    )

    train_dataloader = DataLoader(
        train_dataset,
        batch_size=256,
        shuffle=True,
        num_workers=4,
        persistent_workers=True,
        drop_last=True
    )

    val_dataloader = DataLoader(
        val_dataset,
        batch_size=256,
        shuffle=False,
        num_workers=4,
        persistent_workers=True,
        drop_last=True
    )

    trainer = pl.Trainer(
        max_epochs=config_gae["supp_training_epochs"],
        logger=wandb_logger,
        log_every_n_steps=5,
        accelerator="auto",
        devices=1
    )

    gae_non_cyclic.train()
    trainer.fit(gae_non_cyclic, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)
    gae_non_cyclic.eval()
    wandb.finish()

gen	nevals	valid_recon_ratio	F_ZERO_LEN	F_SUBGROUPS	F_LONG_PARTS	valid_frams_ratio	min   	avg  	max  
0  	140   	1                	0.01      	0.07       	0.64        	0.14             	-0.003	0.195	0.564
1  	140   	0.99             	0         	0.06       	0.71        	0.16             	0.028 	0.268	1.756
2  	140   	1                	0.01      	0.08       	0.71        	0.11             	0.05  	0.192	0.496
3  	140   	0.99             	0         	0.1        	0.71        	0.21             	0.043 	0.188	0.553
4  	140   	1                	0         	0.1        	0.65        	0.2              	0.016 	0.15 	0.4  
5  	140   	1                	0         	0.08       	0.69        	0.22             	0.006 	0.256	0.863
6  	140   	0.99             	0         	0.09       	0.71        	0.19             	0.038 	0.218	1.293
7  	140   	1                	0         	0.06       	0.7         	0.24             	0.002 	0.159	0.403
8  	140   	0.99             	0         	0.09       	0.69        	0.19             

KeyboardInterrupt: 

## VGAE

### Trained once

In [57]:
if IS_NEW_APPROACH:
	vgae_non_cyclic = VariationalGraphAutoencoder(config=config_gae, frams_module=frams)
else:
	vgae_non_cyclic = KlejdaVariationalGraphAutoencoder(config=config_gae, frams_module=frams)
vgae_non_cyclic.load_state_dict(checkpoint_vgae['state_dict'])
vgae_non_cyclic.eval()
evaluator = AutoencoderEvaluator(vgae_non_cyclic, frams_lib, evolution_config['opt_criteria'], evolution_config)
toolbox.register("evaluate", evaluator)

In [10]:
pop, log, reconstruction_ratio, _ = run_cma_es_with_validation(
	toolbox,
	ngen=evolution_config['generations'],
	stats=stats,
	halloffame=hof,
	verbose=True
)

print(f"\nNajlepszy fitness w HoF: {hof[0].fitness.values[0]}")
save_genotypes_json(evolution_config["result_filepath"], hof)

gen	nevals	valid_recon_ratio	F_ZERO_LEN	F_SUBGROUPS	F_LONG_PARTS	valid_frams_ratio	min  	avg  	max  
0  	140   	1                	0         	0          	0           	0.52             	0.263	0.884	1.269
1  	140   	1                	0         	0          	0           	0.62             	0.319	0.877	1.253
2  	140   	1                	0         	0          	0           	0.69             	0.259	0.892	1.322
3  	140   	1                	0         	0          	0           	0.58             	0.267	0.865	1.27 
4  	140   	1                	0         	0          	0           	0.64             	0.209	0.861	1.339
5  	140   	1                	0         	0          	0           	0.59             	0.223	0.842	1.295
6  	140   	1                	0         	0          	0           	0.64             	0.233	0.912	1.344
7  	140   	1                	0         	0          	0           	0.65             	0.311	0.934	1.364
8  	140   	1                	0         	0          	0           	0.69             	0.22 	0.

KeyboardInterrupt: 

### Continual training

Wersja z douczaniem rozdziela się na różne możliwości:
- Stały rozmiar zbioru douczającego, w którym osobniki:
	- Podmieniane są na zasadzie: słabsze odpadają
	- Podmieniane są na zasadzie: starsze odpadają
	- Podmieniane są na zasadzie: losowe odpadają
- Zwiększający się rozmiar zbioru douczającego

In [58]:
loaded_genotypes = []
with open("../results/sampled_best_individuals_new_mini_merged.jsonl", "r", encoding="utf-8") as f:
	for line in f:
		obj = json.loads(line.strip())
		loaded_genotypes.append(obj)

genotypes = deepcopy(loaded_genotypes)
BUFFER_MAX_SIZE = len(genotypes)

for i in range(3):
	toolbox = prepare_cmaes_toolbox(frams_lib, evolution_config)
	toolbox.register("evaluate", evaluator)

	pop, log, created_individuals = run_cma_es_with_validation(
		toolbox,
		ngen=evolution_config['generations'],
		stats=stats,
		halloffame=hof,
		verbose=True,
	)

	new_inds = [{'genotype': ind.genotype, 'fitness': list(ind.fitness.values)} for ind in created_individuals]

	print(f"Dodatkowi osobnicy: {len(new_inds)}")
	fitnesses = [ind['fitness'] for ind in genotypes]
	print(f"Statystyki przed: min: {min(fitnesses)}, max: {max(fitnesses)}, mean: {sum(fitnesses)/len(fitnesses)}")

	genotypes = update_genotypes_buffer(
		current_genotypes=genotypes,
		new_genotypes=new_inds,
		mode=CONTINUAL_TRAINING_DATASET_VERSION,
		max_size=BUFFER_MAX_SIZE
	)
	fitnesses = [ind['fitness'] for ind in genotypes]
	print(f"Statystyki po: min: {min(fitnesses)}, max: {max(fitnesses)}, mean: {sum(fitnesses)/len(fitnesses)}")



	dataset = FramsticksGraphDataset(genotypes, config_vgae["max_nodes"])

	dataset_size = len(dataset)
	train_size = int(0.8 * dataset_size)
	val_size = dataset_size - train_size
	train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

	wandb_logger = WandbLogger(
		project="Framsticks-VGAE",
		name=f"VGAE-Continual-{CONTINUAL_TRAINING_DATASET_VERSION}-Iter-{i}",
		save_dir=config_vgae["save_dir"]
	)

	train_dataloader = DataLoader(
		train_dataset,
		batch_size=256,
		shuffle=True,
		num_workers=4,
		persistent_workers=True,
		drop_last=True
	)

	val_dataloader = DataLoader(
		val_dataset,
		batch_size=256,
		shuffle=False,
		num_workers=4,
		persistent_workers=True,
		drop_last=True
	)

	trainer = pl.Trainer(
		max_epochs=config_vgae["supp_training_epochs"],
		logger=wandb_logger,
		log_every_n_steps=5,
		accelerator="auto",
		devices=1
	)

	vgae_non_cyclic.train()
	trainer.fit(vgae_non_cyclic, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)
	vgae_non_cyclic.eval()
	wandb.finish()

Exception ignored in: <function ExtValue.__del__ at 0x000001DC8861CE00>
Traceback (most recent call last):
  File "C:\Users\witek\PycharmProjects\Magisterka\external\framspy\frams_extvalue.py", line 40, in __del__
    c_api.extFree(self.__ptr)
                  ^^^^^^^^^^
  File "C:\Users\witek\PycharmProjects\Magisterka\external\framspy\frams_extvalue.py", line 212, in __getattr__
    return self.__dict__[key]
           ~~~~~~~~~~~~~^^^^^
KeyError: '_ExtValue__ptr'


gen	nevals	valid_recon_ratio	F_ZERO_LEN	F_SUBGROUPS	F_LONG_PARTS	valid_frams_ratio	min  	avg  	max  
0  	140   	1                	0         	0          	0           	0.03             	0.063	0.195	0.319
1  	140   	1                	0         	0          	0           	0.02             	0.142	0.269	0.452
2  	140   	1                	0         	0          	0           	0.04             	0.117	0.217	0.347
3  	140   	1                	0         	0          	0           	0.06             	0.131	0.209	0.416
4  	140   	1                	0         	0          	0           	0.04             	0.156	0.195	0.322
5  	140   	1                	0         	0          	0           	0.09             	0.057	0.178	0.385
6  	140   	1                	0         	0          	0           	0.06             	0.04 	0.126	0.281
7  	140   	1                	0         	0          	0           	0.06             	0.061	0.149	0.24 
8  	140   	1                	0         	0          	0           	0.06             	0.107	0.

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
C:\Users\witek\PycharmProjects\Magisterka\.venv\Lib\site-packages\pytorch_lightning\loggers\wandb.py:400: There is a wandb run already in progress and newly created instances of `WandbLogger` will reuse this run. If this is not desired, call `wandb.finish()` before instantiating `WandbLogger`.
C:\Users\witek\PycharmProjects\Magisterka\.venv\Lib\site-packages\pytorch_lightning\callbacks\model_checkpoint.py:881: Checkpoint directory ./checkpoints\Framsticks-VGAE\6gwlegpu\checkpoints

99 	140   	1                	0         	0          	0           	0.82             	0.196	0.858	1.383


┏━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name             ┃ Type     ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ encoder_backbone │ Encoder  │  2.1 M │ train │     0 │
│ 1 │ fc_mu            │ Linear   │  1.9 K │ train │     0 │
│ 2 │ fc_logvar        │ Linear   │  1.9 K │ train │     0 │
│ 3 │ decoder_a        │ DecoderA │ 93.7 K │ train │     0 │
│ 4 │ decoder_x        │ DecoderX │  562 K │ train │     0 │
└───┴──────────────────┴──────────┴────────┴───────┴───────┘

...

┏━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name             ┃ Type     ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ encoder_backbone │ Encoder  │  2.1 M │ train │     0 │
│ 1 │ fc_mu            │ Linear   │  1.9 K │ train │     0 │
│ 2 │ fc_logvar        │ Linear   │  1.9 K │ train │     0 │
│ 3 │ decoder_a        │ DecoderA │ 93.7 K │ train │     0 │
│ 4 │ decoder_x        │ DecoderX │  562 K │ train │     0 │
└───┴──────────────────┴──────────┴────────┴───────┴───────┘

Trainable params: 2.8 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 2.8 M                                                                                                
Total estimated model params size (MB): 11.076                                                                     
Modules in train mode: 65                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

`Trainer.fit` stopped: `max_epochs=16` reached.


wandb: updating run metadata
wandb: uploading wandb-summary.json; uploading config.yaml
wandb: uploading history steps 30-31, summary
wandb: 
wandb: Run history:
wandb:                      epoch ▁▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇████
wandb: train/locality_correlation ▁▃▄▅▅▅▆▆▆▆▇▇█▇██
wandb:               train/loss_A █▆▅▅▄▄▄▃▃▃▂▂▂▂▁▁
wandb:              train/loss_KL ▃▁▁▂▂▂▂▄▄▄▅▆▆▆██
wandb:               train/loss_X █▆▆▅▄▄▃▃▃▃▂▂▂▂▁▁
wandb:        train/loss_locality █▆▅▄▄▄▃▃▃▃▂▂▁▂▁▁
wandb:           train/loss_total █▆▅▅▄▄▃▃▃▃▂▂▂▂▁▁
wandb:          train/metric_A_F1 ▁▃▃▄▄▅▅▆▆▆▇▇▇▇██
wandb:    train/metric_A_fp_ratio █▇▆▆▅▅▄▄▃▃▃▂▂▂▁▁
wandb:      train/metric_A_g_mean ▁▃▄▄▅▅▅▆▆▆▇▇▇▇██
wandb:                        +16 ...
wandb: 
wandb: Run summary:
wandb:                      epoch 15
wandb: train/locality_correlation 0.74997
wandb:               train/loss_A 0.27158
wandb:              train/loss_KL 60.5191
wandb:               train/loss_X 0.02656
wandb:        train/loss_locality 0.0005


# Results